# Breast Cancer Wisconsin: Research Visualization Teaching Notebook

This notebook is designed for **beginners** who may not know Python deeply yet, but need to learn how to create **research-style visualizations** from a machine learning experiment.

We will:

1. Load and inspect the **Breast Cancer Wisconsin (Diagnostic)** dataset
2. Split and preprocess the data
3. Train **three models**
   - Logistic Regression
   - Random Forest
   - Neural Network (MLPClassifier)
4. Evaluate each model
5. Create a structure where **students add visualizations**
6. Compare model performance in a way similar to a mini research workflow

---

## Teaching goals

By the end of this notebook, students should be able to:

- understand the shape of a dataset
- understand binary classification outputs
- visualize model training and evaluation
- produce publication-style figures
- write short research-style interpretations for each figure

---

## Important note for students

You are **allowed to use AI tools**, but you must still understand:

- what each graph is showing
- why that graph is appropriate
- what conclusion can and cannot be made from it

## Notebook flow

The notebook alternates between:

- **code cells you run**
- **explanation cells**
- **student task placeholders** where students must add graphs and interpretations

This is intentional.  
The point is not only to train models, but to turn model results into **research communication**.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    classification_report,
    log_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

### What this section does

We import the libraries required for:

- loading the dataset
- preprocessing
- training models
- evaluating models
- plotting results

---

### Student note

You do **not** need to memorize all imports right now.  
What matters is understanding which outputs from the model can later be turned into figures.

## 2. Load the Breast Cancer Wisconsin dataset

In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

class_names = data.target_names
feature_names = data.feature_names

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Class names:", class_names)

### What this dataset contains

This is a **binary classification** dataset.

- Each row is a patient sample
- Each column is a numeric feature computed from diagnostic measurements
- The target has **2 classes**
  - 0 = malignant
  - 1 = benign

Binary classification is extremely useful for teaching because it allows:

- confusion matrices
- ROC curves
- Precision-Recall curves
- threshold thinking
- probability-based interpretation

## 3. Create a full DataFrame for inspection

In [ ]:
df = X.copy()
df["target"] = y
df["target_label"] = df["target"].map({0: "malignant", 1: "benign"})

df.head()

### Why this is useful

Combining the feature matrix and target into one DataFrame makes it easier to perform:

- exploratory analysis
- class-wise visualization
- grouped plotting
- correlation analysis

## 4. Basic dataset inspection

In [ ]:
print("First 5 rows:")
display(df.head())

print("\nMissing values per column:")
display(df.isnull().sum().sort_values(ascending=False).head())

print("\nSummary statistics:")
display(df.describe())

# Student Visualization Task 1: Dataset overview

Create the following visualizations below this markdown cell:

1. **Class distribution bar chart**
2. **Histogram of one important feature**
3. **Box plot of one feature by class**
4. **Correlation heatmap** for selected features or full numerical features

### What students should learn here

Before training any model, a researcher should understand:

- how balanced the classes are
- how features are distributed
- whether classes appear separable
- whether features are strongly correlated

### Suggested research-style questions

- Is the dataset class balanced?
- Do some features differ clearly between malignant and benign samples?
- Are some features highly correlated?

In [ ]:
# STUDENT WORK AREA 1
# Add your dataset visualizations here

# Student Interpretation Task 1

Write 4-6 lines below describing what your dataset visualizations show.

Suggested structure:

- What does the class distribution show?
- Which feature distributions seem informative?
- Is there visible separation between classes?
- What does the heatmap suggest about redundancy/correlation in features?

## 5. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

### Why we split the data

We train the model on one part of the data and evaluate it on another part.

This helps answer:

**Does the model generalize to unseen data?**

The split is stratified so that class proportions remain similar in train and test sets.

## 6. Standardize data for models that need scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

### Why scaling matters

Some models are sensitive to feature scale.

Especially:

- Logistic Regression
- Neural Networks

Random Forest generally does not require scaling in the same way.

## 7. Train Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=5000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred_log = log_reg.predict(X_test_scaled)
y_prob_log = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression trained successfully.")

### Why Logistic Regression is included

Logistic Regression is a very important baseline model because it is:

- fast
- interpretable
- probability-based
- excellent for ROC and Precision-Recall teaching

## 8. Train Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest trained successfully.")

### Why Random Forest is included

Random Forest gives us:

- a strong classical baseline
- probability predictions
- feature importance
- a different modeling philosophy than linear models

## 9. Train Neural Network (MLPClassifier)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20
)
mlp.fit(X_train_scaled, y_train)

y_pred_mlp = mlp.predict(X_test_scaled)
y_prob_mlp = mlp.predict_proba(X_test_scaled)[:, 1]

print("Neural Network trained successfully.")
print("Number of training iterations actually used:", mlp.n_iter_)

### Why the neural network is included

The neural network is useful because it produces the kind of training behavior students often see in research papers:

- loss across iterations
- learning dynamics
- validation behavior

This makes it ideal for teaching:
- training curves
- overfitting discussion
- optimization behavior

## 10. Create a helper function for metrics

In [ ]:
def get_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred),
        "ROC AUC": roc_auc_score(y_true, y_prob),
        "Average Precision": average_precision_score(y_true, y_prob),
        "Log Loss": log_loss(y_true, y_prob)
    }

## 11. Compute metrics for all models

In [ ]:
results = pd.DataFrame({
    "Logistic Regression": get_metrics(y_test, y_pred_log, y_prob_log),
    "Random Forest": get_metrics(y_test, y_pred_rf, y_prob_rf),
    "Neural Network": get_metrics(y_test, y_pred_mlp, y_prob_mlp)
}).T

results = results.sort_values(by="ROC AUC", ascending=False)
results

### Why this table matters

A research workflow does not stop at one metric.

Students should learn that model quality can be compared using:

- Accuracy
- Precision
- Recall
- F1 Score
- ROC AUC
- Average Precision
- Log Loss

Each metric tells a slightly different story.

# Student Visualization Task 2: Model comparison summary

Create visualizations below this markdown cell for:

1. **Bar chart comparing Accuracy, Precision, Recall, and F1 across models**
2. **Bar chart comparing ROC AUC across models**
3. **Bar chart comparing Log Loss across models**

### What students should learn here

A single scalar table is useful, but figures communicate comparisons more clearly in research reports.

In [ ]:
# STUDENT WORK AREA 2
# Add model comparison visualizations here

# Student Interpretation Task 2

Write 4-6 lines interpreting the model comparison figures.

Suggested questions:

- Which model performs best overall?
- Do all metrics point to the same conclusion?
- Is any model stronger on one metric but weaker on another?
- Why might that happen?

## 12. Print classification reports

In [ ]:
print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_log, target_names=class_names))

print("\n=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, target_names=class_names))

print("\n=== Neural Network ===")
print(classification_report(y_test, y_pred_mlp, target_names=class_names))

### Why classification reports matter

These reports help students connect the summary metrics with class-wise behavior.

For example:

- is one class being predicted better than the other?
- is recall strong but precision weaker?
- is the model conservative or aggressive in positive predictions?

## 13. Confusion matrices for all models

In [ ]:
cm_log = confusion_matrix(y_test, y_pred_log)
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_mlp = confusion_matrix(y_test, y_pred_mlp)

print("Logistic Regression confusion matrix:\n", cm_log)
print("\nRandom Forest confusion matrix:\n", cm_rf)
print("\nNeural Network confusion matrix:\n", cm_mlp)

# Student Visualization Task 3: Confusion matrices

Create **three confusion matrix visualizations** below this cell:

- Logistic Regression confusion matrix
- Random Forest confusion matrix
- Neural Network confusion matrix

### Suggested expectations

Students should label:
- axes
- class names
- counts clearly

### Interpretation prompt

Discuss:
- false positives
- false negatives
- which errors matter more in a medical context

In [ ]:
# STUDENT WORK AREA 3
# Add confusion matrix visualizations here

# Student Interpretation Task 3

Write a short paragraph explaining the confusion matrices.

Important research-thinking question:

In medical diagnosis, which is often worse:
- false positive?
- false negative?

Why?

## 14. ROC curve data

In [ ]:
fpr_log, tpr_log, _ = roc_curve(y_test, y_prob_log)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, y_prob_mlp)

roc_auc_log = roc_auc_score(y_test, y_prob_log)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)
roc_auc_mlp = roc_auc_score(y_test, y_prob_mlp)

print("ROC AUC - Logistic Regression:", roc_auc_log)
print("ROC AUC - Random Forest:", roc_auc_rf)
print("ROC AUC - Neural Network:", roc_auc_mlp)

# Student Visualization Task 4: ROC curves

Create a single figure below this markdown cell containing ROC curves for all three models.

Requirements:

- one line per model
- diagonal random baseline
- legend with AUC values
- title and labeled axes

### What students should learn

ROC curves show performance across thresholds, not just at one fixed decision cutoff.

In [ ]:
# STUDENT WORK AREA 4
# Add ROC curve visualization here

# Student Interpretation Task 4

Write 3-5 lines explaining the ROC curves.

Suggested questions:

- Which model dominates the ROC space?
- Are the differences large or small?
- Why is ROC useful beyond plain accuracy?

## 15. Precision-Recall curve data

In [ ]:
precision_log, recall_log, _ = precision_recall_curve(y_test, y_prob_log)
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_prob_rf)
precision_mlp, recall_mlp, _ = precision_recall_curve(y_test, y_prob_mlp)

ap_log = average_precision_score(y_test, y_prob_log)
ap_rf = average_precision_score(y_test, y_prob_rf)
ap_mlp = average_precision_score(y_test, y_prob_mlp)

print("Average Precision - Logistic Regression:", ap_log)
print("Average Precision - Random Forest:", ap_rf)
print("Average Precision - Neural Network:", ap_mlp)

# Student Visualization Task 5: Precision-Recall curves

Create a single figure below this markdown cell containing Precision-Recall curves for all three models.

Requirements:

- one line per model
- legend with average precision values
- title and labeled axes

### Why this matters

Precision-Recall curves are often especially useful when class balance or positive-class focus matters.

In [ ]:
# STUDENT WORK AREA 5
# Add Precision-Recall curve visualization here

# Student Interpretation Task 5

Write 3-5 lines explaining the Precision-Recall curves.

Suggested questions:

- Which model maintains better precision as recall increases?
- How does this complement ROC analysis?

## 16. Neural network training history

In [ ]:
mlp_loss_curve = mlp.loss_curve_
print("Number of recorded loss values:", len(mlp_loss_curve))
print("First 10 loss values:", mlp_loss_curve[:10])

### Important note

`MLPClassifier` gives us a loss curve through `loss_curve_`.

This is useful for teaching:
- optimization progress
- whether the model is still improving
- whether training stabilized

Because `early_stopping=True`, the classifier also tracked validation behavior internally, but scikit-learn does not expose a full train/validation history as cleanly as deep learning libraries do.

Still, the loss curve is very useful pedagogically.

# Student Visualization Task 6: Neural network training curve

Create a line plot of the neural network loss across iterations.

Requirements:

- x-axis = iteration
- y-axis = loss
- proper title and labels

### Advanced extension

If you want, also annotate where the curve seems to flatten.

In [ ]:
# STUDENT WORK AREA 6
# Add neural network loss curve visualization here

# Student Interpretation Task 6

Write 3-5 lines explaining the training curve.

Suggested questions:

- Is loss decreasing steadily?
- Does training appear to stabilize?
- Is there visible evidence of convergence?

## 17. Feature importance / coefficient information

In [ ]:
log_coefficients = pd.DataFrame({
    "feature": feature_names,
    "coefficient": log_reg.coef_[0],
    "abs_coefficient": np.abs(log_reg.coef_[0])
}).sort_values("abs_coefficient", ascending=False)

rf_importances = pd.DataFrame({
    "feature": feature_names,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

display(log_coefficients.head(10))
display(rf_importances.head(10))

# Student Visualization Task 7: Model interpretation visuals

Create the following:

1. **Top 10 absolute coefficients for Logistic Regression**
2. **Top 10 feature importances for Random Forest**

### What students should learn

Different models provide different ways to understand what drives predictions.

This is a crucial research skill:
not just asking **how accurate is the model?**
but also asking **what seems to influence the model?**

In [ ]:
# STUDENT WORK AREA 7
# Add feature importance / coefficient plots here

# Student Interpretation Task 7

Write 4-6 lines comparing Logistic Regression coefficients and Random Forest importances.

Suggested questions:

- Do the same features appear important in both models?
- Why might rankings differ?
- What is the difference between a coefficient and a feature importance score?

## 18. Probability distributions by true class

In [ ]:
prob_df = pd.DataFrame({
    "true_label": y_test.values,
    "true_label_name": pd.Series(y_test.values).map({0: "malignant", 1: "benign"}),
    "logistic_prob_benign": y_prob_log,
    "rf_prob_benign": y_prob_rf,
    "mlp_prob_benign": y_prob_mlp
})

display(prob_df.head())

# Student Visualization Task 8: Probability score distributions

Create probability distribution plots (histogram or KDE-based plot) for at least one model showing:

- predicted probability for benign class
- grouped by true class

You may create this for:
- Logistic Regression
- Random Forest
- Neural Network
- or all three

### Why this matters

This helps students see whether the model separates classes confidently or ambiguously.

In [ ]:
# STUDENT WORK AREA 8
# Add probability distribution visualizations here

# Student Interpretation Task 8

Write 3-5 lines explaining what the predicted probability distributions show.

Suggested questions:

- Are the two classes well separated?
- Are there overlapping regions?
- What does overlap imply?

## 19. Identify misclassified examples

In [ ]:
misclassified_log = X_test.copy()
misclassified_log["true"] = y_test.values
misclassified_log["pred"] = y_pred_log
misclassified_log["prob_benign"] = y_prob_log
misclassified_log = misclassified_log[misclassified_log["true"] != misclassified_log["pred"]]

misclassified_rf = X_test.copy()
misclassified_rf["true"] = y_test.values
misclassified_rf["pred"] = y_pred_rf
misclassified_rf["prob_benign"] = y_prob_rf
misclassified_rf = misclassified_rf[misclassified_rf["true"] != misclassified_rf["pred"]]

misclassified_mlp = X_test.copy()
misclassified_mlp["true"] = y_test.values
misclassified_mlp["pred"] = y_pred_mlp
misclassified_mlp["prob_benign"] = y_prob_mlp
misclassified_mlp = misclassified_mlp[misclassified_mlp["true"] != misclassified_mlp["pred"]]

print("Misclassified samples - Logistic Regression:", len(misclassified_log))
print("Misclassified samples - Random Forest:", len(misclassified_rf))
print("Misclassified samples - Neural Network:", len(misclassified_mlp))

# Student Visualization Task 9: Misclassification analysis

Create one or more visualizations showing model errors.

Possible ideas:

- bar chart of number of misclassified samples per model
- scatter plot of two selected features highlighting misclassified points
- histogram of predicted probabilities for misclassified samples

### Why this matters

Research is not only about success.  
It is also about understanding failure patterns.

In [ ]:
# STUDENT WORK AREA 9
# Add misclassification analysis plots here

# Student Interpretation Task 9

Write a short paragraph discussing misclassifications.

Suggested questions:

- Which model makes the fewest errors?
- Are the errors low-confidence or high-confidence mistakes?
- Why is error analysis important in real applications?

## 20. Optional advanced visualization ideas

Students or instructors may optionally add:

1. PCA 2D projection colored by class
2. PCA projection highlighting model mistakes
3. threshold vs precision / recall / F1 for one model
4. calibration curve
5. pair plot for selected features
6. comparison dashboard figure combining multiple subplots

These are excellent extensions for stronger students.

In [ ]:
# OPTIONAL STUDENT WORK AREA 10
# Add advanced visualizations here

# Final Student Research Summary

Below this markdown cell, students should write a short mini-results section answering:

1. Which model performed best overall?
2. Which visualizations were most informative?
3. What did ROC and Precision-Recall curves reveal?
4. What did error analysis reveal?
5. What are the limitations of this experiment?

This final section should read like a small **research report summary**, not casual commentary.

## Instructor note

This notebook is intentionally structured to support:

- beginners
- AI-assisted coding
- research-style thinking
- figure creation and interpretation

A strong classroom rule would be:

> You may use AI for plotting code, but you must justify the chart choice and explain the result in your own words.